# Gaussova (normalna) raspodjela: Galtonova daska (knjiga, poglavlje 2.4.2)

$$p(x)=\frac{1}{\sqrt{2\pi}\,\sigma}e^{-\frac{(x-a)^2}{2\sigma^2}},\qquad
P(x)=\frac12\Big[1+\mathrm{erf}\Big(\frac{x-a}{\sqrt2\,\sigma}\Big)\Big]\qquad(2.12),(2.13)$$

**Eksperiment.** Galtonova daska je kosa daska sa $R$ redova klinova. Kuglica pada odozgo i na svakom klinu skrene
lijevo ili desno, svaki put sa vjerovatnoćom $1/2$. Na dnu se kuglice skupljaju u pregrade i formiraju **zvono**.

Konačni položaj kuglice je zbir $R$ nezavisnih skretanja $\pm1$: $\;X=Y_1+Y_2+\dots+Y_R$.
To je tačno situacija iz poglavlja 3.5 knjige (centralna granična teorema): zbir mnogo malih nezavisnih uticaja
ima Gaussovu raspodjelu. Isto objašnjava zašto su visina ljudi, greške mjerenja i šum približno Gaussovi.

Za jedno skretanje vrijedi $E[Y]=0$ i $E[Y^2]=1$, a karakteristična funkcija je $\Phi_Y(u)=\frac{e^{ju}+e^{-ju}}{2}=\cos u$. Zato je

$$\Phi_{X/\sqrt R}(u)=\Big[\cos\frac{u}{\sqrt R}\Big]^R=\Big[1-\frac{u^2}{2R}+\dots\Big]^R\ \longrightarrow\ e^{-u^2/2},$$

a to je karakteristična funkcija standardne normalne raspodjele. Dakle $X\approx\mathcal N(0,\,R)$, tj. $a=0$ i $\sigma=\sqrt R$.

**Pokretanje:** u Google Colabu `Runtime → Run all`. Prva ćelija sama klonira repozitorij
[esokic/ssp](https://github.com/esokic/ssp) i učita biblioteku `ssp` koja prati knjigu.
Svaka bilježnica radi $N=10\,000$ eksperimenata. Detaljna izvođenja su u PDF-u.

In [ ]:
# Priprema okruženja: kloniranje repozitorija sa SSP bibliotekom (prema README uputstvu)
import os, sys, subprocess

REPO_URL = "https://github.com/ierovic1ba/ssp.git"   # fork originala https://github.com/esokic/ssp
SSP_PUTANJA = "/content/ssp" if os.path.isdir("/content") else os.path.abspath("ssp")

if not os.path.isdir(os.path.join(SSP_PUTANJA, "ssp")):
    subprocess.run(["git", "clone", REPO_URL, SSP_PUTANJA], check=True)

sys.path.insert(0, SSP_PUTANJA)   # folder u kojem se nalazi paket ssp/
import ssp

import numpy as np
import matplotlib.pyplot as plt

# Podešavanje prikaza za rad studenata (preporuka iz README-a):
ssp.FORMATIRANJE_PRIKAZA = "JUPYTER"
ssp.REZOLUCIJA = "LOW"

N = 10000   # broj eksperimenata

## Kako izgleda Galtonova daska
Crne tačke su klinovi, a obojene linije su putanje nekoliko kuglica.

In [ ]:
R = 12                                           # broj redova klinova
plt.figure(figsize=(6, 5))
for red in range(R):                             # klinovi: red i ima i+1 klinova
    plt.plot(np.arange(-red, red + 1, 2), -red*np.ones(red + 1), "k.", ms=6)
for _ in range(6):                               # putanje 6 kuglica
    skretanja = np.random.choice([-1, 1], size=R)
    putanja = np.concatenate([[0], np.cumsum(skretanja)])
    plt.plot(putanja, -np.arange(R + 1), "-", lw=2, alpha=0.8)
plt.title(f"Galtonova daska sa {R} redova klinova"); plt.xlabel("položaj"); plt.yticks([])
plt.show()

## Puštamo 10 000 kuglica
Za mali broj redova raspodjela je „uglasta“ (binomijalna), a sa više redova sve više liči na Gaussovo zvono.
Položaj je standardizovan sa $\sigma=\sqrt R$ da bi se svi grafici mogli porediti sa $\mathcal N(0,1)$.

In [ ]:
z = np.linspace(-4, 4, 1000)
phi = np.exp(-z**2 / 2) / np.sqrt(2*np.pi)      # relacija (2.8)

fig, ax = plt.subplots(2, 2, figsize=(10, 6))
for osa, R in zip(ax.flat, [2, 5, 12, 50]):
    skretanja = np.random.choice([-1, 1], size=(N, R))   # N kuglica, R klinova
    X = skretanja.sum(axis=1)                            # konačni položaj
    ivice = (np.arange(-R - 1, R + 2, 2)) / np.sqrt(R)   # jedna pregrada = jedan stubić
    osa.hist(X / np.sqrt(R), bins=ivice, density=True, alpha=0.7, label="kuglice")
    osa.plot(z, phi, "r", lw=2, label="N(0,1)")
    osa.set_xlim(-4, 4); osa.set_title(f"R = {R} redova klinova"); osa.grid(alpha=0.3)
ax[1, 1].legend()
fig.suptitle("Galtonova daska: standardizovani položaj kuglica")
fig.tight_layout(); plt.show()

## Detaljna provjera za $R=50$ redova
Teorija: $a=0$, $\sigma^2=R=50$. Rezultat se poredi i sa `np.random.normal` iz programskog koda 2.2.

In [ ]:
from math import erf
R = 50
X = np.random.choice([-1, 1], size=(N, R)).sum(axis=1)
a, sigma = 0, np.sqrt(R)

ssp.slprom1D.realizacije(np.arange(500), X[:500], np.arange(500), float(a),
                         legenda=["kuglice", "$a$"], naslov="Položaj prvih 500 kuglica ($R=50$)")

x = np.linspace(-4*sigma, 4*sigma, 1000)
p_teor = np.exp(-(x - a)**2 / (2*sigma**2)) / (np.sqrt(2*np.pi) * sigma)   # relacija (2.12)
X_knjiga = np.random.normal(a, sigma, N)                                   # programski kod 2.2

plt.figure(figsize=(6, 3.5))
plt.hist(X, bins=np.arange(-R - 1, R + 2, 2), density=True, alpha=0.6, label="Galtonova daska")
plt.hist(X_knjiga, bins=40, density=True, histtype="step", lw=1.5, color="k", label="np.random.normal")
plt.plot(x, p_teor, "r", lw=2, label="teorija (2.12)")
plt.xlim(-4*sigma, 4*sigma); plt.title(f"Funkcija gustine, $a=0$, $\\sigma=\\sqrt{{{R}}}$")
plt.xlabel("položaj kuglice"); plt.legend(); plt.grid(alpha=0.3); plt.show()

xs, Px = ssp.slprom1D.PDF(X, step=0.25, prosirenje_opsega=0.0, prikaz=False)
Px_teor = 0.5 * (1 + np.array([erf((t - a) / (np.sqrt(2)*sigma)) for t in xs]))  # relacija (2.13)
ssp.slprom1D.grafik(xs, Px, xs, Px_teor, stil2="r--",
                    x_labela="$x$", naslov="Funkcija raspodjele vjerovatnoće $P(x)$",
                    legenda=["Galtonova daska", "teorija (2.13)"], lokacija_lgd="lower right")

print(f"E[X]   = {np.mean(X):.4f}   (teorija a       = {a})")
print(f"Var[X] = {np.var(X):.4f}   (teorija sigma^2 = R = {R})")
print(f"P(|X-a| <= 2 sigma) = {np.mean(abs(X - a) <= 2*sigma):.4f}   (teorija 0.9545)")
print(f"P(|X-a| <= 3 sigma) = {np.mean(abs(X - a) <= 3*sigma):.4f}   (teorija 0.9973)")

**Napomena:** kuglica može završiti samo na parnim (za parno $R$) položajima, pa histogram sa „pregradama“ širine 2
tačno odgovara stvarnoj Galtonovoj dasci. Pravilo $68$–$95$–$99{,}7\%$ za $\pm1,2,3\sigma$ vrijedi za svaku Gaussovu raspodjelu.
Udio unutar $\pm2\sigma$ ovdje izlazi oko $0{,}967$ umjesto $0{,}9545$. Razlog je diskretnost: granica $2\sigma\approx14{,}1$
obuhvati cijele pregrade $\pm14$ (tačna binomijalna vjerovatnoća je $0{,}967$). Sa više redova ta razlika nestaje.